# 05 – Amino-acid enrichment against a background population

**Question:** which amino acids are over- or under-represented in a group of peptides (Natural, Synthetic, and peptides listed with the Gram+, Gram-, Fungus, Cancer, Virus and Mammalian-cell target groups) compared with a background?

**Input:** the **non-redundant** peptides from notebook 02 (CD-HIT representatives, 90% identity), so that large families of near-identical peptides do not dominate the residue counts.

## Method: Composition Profiler
We follow the method of **Composition Profiler** (Vacic et al. 2007), a tool for "semi-automatic discovery of enrichment or depletion of amino acids". It takes a *query sample* (our group) and a *reference sample* that "provides a suitable background amino acid distribution" and that may be *"a standard protein database (e.g. SwissProt)"*, *"a representative sample of proteins from the organism under study"*, or *"a group of proteins with a contrasting functional annotation"*. DBAASP's own composition tool uses the same idea, comparing peptides with all UniProt proteins (Pirtskhalava et al. 2016, 2021).

For each amino acid *k*, with *p_k* = its relative frequency in the group and *q_k* = its relative frequency in the background:

| Quantity | Definition (Vacic et al. 2007) |
|---|---|
| **fractional difference** *h_k* | (p_k − q_k) / q_k. 0 = same as background, +1 = twice as common, −0.5 = half as common |
| how *p_k*, *q_k* are estimated | means over **bootstrap replicates that resample whole peptides** (1,000 here) from each sample |
| **error bar** *e_k* | ((p_k + σ_p) − (q_k + σ_q)) / (q_k + σ_q) − h_k, where σ are the standard deviations over the bootstrap replicates |
| **significance** | a **two-sample t-test between two sequences of binary indicator variables** (1 if the residue is this amino acid, 0 if not), one sequence per sample; the optional conservative **Bonferroni correction** divides α = 0.05 by the number of tests (20 amino acids) |

We use **two backgrounds**:

1. **The other peptides** in the non-redundant set (all peptides *outside* the group): the "contrasting" sample. This gets the t-test.
2. **Swiss-Prot**: the amino-acid composition of "SwissProt 51" from Table 1 of Vacic et al. (2007) (copied below). Here *q_k* is a fixed table, so we report *h_k* and its error bar but no p-value (the paper's test needs two samples).

**Things to keep in mind**
* Like the original method, the t-test treats every residue as independent, but residues of one peptide are not independent, so the p-values are optimistic. The bootstrap over whole peptides (the error bars) shows the more realistic uncertainty.
* With thousands of residues, almost every difference is statistically significant; look at the **size** of *h_k* and its error bar, not only at the stars.
* DBAASP notes that compositional signatures against UniProt are most meaningful for *ribosomal* peptides, which are evolved sequences (Pirtskhalava et al. 2021). Synthetic peptides are designed, so their difference from Swiss-Prot mostly reflects design choices.
* A "group" means the peptide is *listed with* that target group in DBAASP (not necessarily active against it; Pirtskhalava et al. 2021). Groups overlap, but each is compared separately with its own background.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind_from_stats

df = pd.read_csv("../results/peptide_features.csv")
nr = df[df["is_representative"]].reset_index(drop=True)
print(f"{len(df)} cleaned peptides -> {len(nr)} non-redundant peptides used here")

AA = list("ACDEFGHIKLMNPQRSTVWY")
# amino acids ordered by chemical class for plotting: basic | acidic | polar | special | hydrophobic / aromatic
AA_PLOT = list("KRH") + list("DE") + list("STNQ") + list("CGP") + list("AVLIM") + list("FWY")
assert sorted(AA_PLOT) == sorted(AA)
N_BOOT = 1000
ALPHA = 0.05

# residue counts per peptide: N peptides x 20 amino acids
counts = np.array([[s.count(a) for a in AA] for s in nr["sequence"]])
assert (counts.sum(axis=1) == nr["length"].to_numpy()).all()
counts.shape

18275 cleaned peptides -> 9951 non-redundant peptides used here


(9951, 20)

## Swiss-Prot background
Amino-acid composition (%) of **SwissProt 51**, copied from Table 1 of Vacic et al. (2007) (the paper lists mean ± SD from 100,000 bootstrap iterations; we use the means). Swiss-Prot release 51 is old (2006), which is fine for a rough proteome-wide reference.

In [2]:
SWISSPROT_PCT = {"A": 7.89, "R": 5.40, "N": 4.13, "D": 5.35, "C": 1.50, "Q": 3.95, "E": 6.67, "G": 6.96,
                 "H": 2.29, "I": 5.90, "L": 9.65, "K": 5.92, "M": 2.38, "F": 3.96, "P": 4.83, "S": 6.83,
                 "T": 5.41, "W": 1.13, "Y": 3.03, "V": 6.73}          # Vacic et al. (2007), Table 1
assert set(SWISSPROT_PCT) == set(AA)
assert abs(sum(SWISSPROT_PCT.values()) - 100) < 0.2                     # the table is rounded, so it sums to ~100
swissprot = np.array([SWISSPROT_PCT[a] for a in AA]) / sum(SWISSPROT_PCT.values())

all_counts = counts.sum(axis=0)
pd.DataFrame({"Swiss-Prot 51 (%)": pd.Series(SWISSPROT_PCT),
              "all our peptides (%)": pd.Series(100 * all_counts / all_counts.sum(), index=AA)}).loc[AA_PLOT].round(2).T

,K,R,H,D,E,S,T,N,Q,C,G,P,A,V,L,I,M,F,W,Y
Swiss-Prot 51 (%),5.92,5.40,2.29,5.35,6.67,6.83,5.41,4.13,3.95,1.50,6.96,4.83,7.89,6.73,9.65,5.90,2.38,3.96,1.13,3.03
all our peptides (%),14.31,9.87,2.08,1.81,2.05,4.22,2.89,2.67,2.28,3.62,7.67,3.82,6.99,5.51,11.15,6.39,1.08,5.09,4.39,2.11


## Groups

In [3]:
groups = {
    "Natural": nr["origin"] == "Natural",
    "Synthetic": nr["origin"] == "Synthetic",
    "Gram+": nr["target_gram_pos"] == 1,
    "Gram-": nr["target_gram_neg"] == 1,
    "Fungus": nr["target_fungus"] == 1,
    "Cancer": nr["target_cancer"] == 1,
    "Virus": nr["target_virus"] == 1,
    "Mammalian cell": nr["target_mammalian_cell"] == 1,
}
groups = {k: v.to_numpy() for k, v in groups.items()}
pd.DataFrame({"peptides": {k: int(v.sum()) for k, v in groups.items()},
              "% of all peptides": {k: round(100 * v.mean(), 1) for k, v in groups.items()},
              "residues": {k: int(counts[v].sum()) for k, v in groups.items()}})

,peptides,% of all peptides,residues
Natural,2301,23.1,64963
Synthetic,7650,76.9,124181
Gram+,7859,79.0,151981
Gram-,8028,80.7,155361
Fungus,2924,29.4,60276
Cancer,1819,18.3,32161
Virus,682,6.9,14001
Mammalian cell,5469,55.0,99589


## The Composition Profiler calculation

In [4]:
def bootstrap_frequencies(sample_counts, rng, n_boot=N_BOOT):
    """Resample WHOLE peptides with replacement; return the residue-frequency vector of each replicate
    (Vacic et al. 2007: 'bootstrap sampling of whole proteins from the original samples')."""
    n = len(sample_counts)
    freqs = np.empty((n_boot, 20))
    for b in range(n_boot):
        c = sample_counts[rng.integers(0, n, n)].sum(axis=0)
        freqs[b] = c / c.sum()
    return freqs

def profile(fg_counts, bg_counts=None, bg_freq=None, seed=0):
    """Fractional difference h_k and error bar e_k (Vacic et al. 2007) of a group against a background.
    Give the background either as a peptide sample (bg_counts) or as fixed frequencies (bg_freq)."""
    rng = np.random.default_rng(seed)
    pb = bootstrap_frequencies(fg_counts, rng)
    p, sp = pb.mean(axis=0), pb.std(axis=0)
    if bg_counts is not None:
        qb = bootstrap_frequencies(bg_counts, rng)
        q, sq = qb.mean(axis=0), qb.std(axis=0)
    else:
        q, sq = bg_freq, np.zeros(20)
    h = (p - q) / q                                            # fractional difference
    e = ((p + sp) - (q + sq)) / (q + sq) - h                   # error bar
    out = pd.DataFrame({"aa": AA, "pct_group": 100 * p, "pct_background": 100 * q,
                        "fractional_difference": h, "error_bar": e})
    if bg_counts is not None:
        # two-sample t-test between binary indicator sequences ('is this residue amino acid k?'), from summary statistics
        n1, n2 = fg_counts.sum(), bg_counts.sum()
        p0, q0 = fg_counts.sum(0) / n1, bg_counts.sum(0) / n2
        sd1 = np.sqrt(p0 * (1 - p0) * n1 / (n1 - 1))           # sample SD of a 0/1 sequence
        sd2 = np.sqrt(q0 * (1 - q0) * n2 / (n2 - 1))
        pvals = ttest_ind_from_stats(p0, sd1, n1, q0, sd2, n2).pvalue
        out["p_t_test"] = pvals
        out["p_bonferroni"] = np.minimum(pvals * 20, 1)         # Bonferroni: alpha / 20 tests
        out["significant"] = out["p_bonferroni"] < ALPHA
    return out

# --- tests on toy data ---
toy_fg = np.array([[10] + [0] * 19, [10] + [0] * 19])          # group made only of A
toy_bg = np.array([[5] + [5] * 19, [5] + [5] * 19])            # background: every amino acid equally common
r = profile(toy_fg, toy_bg)
assert abs(r.loc[0, "fractional_difference"] - (1 / (1 / 20) - 1)) < 1e-9     # A is 20x more common: h = 19
assert (r.loc[1:, "fractional_difference"] == -1).all()                         # everything else absent: h = -1
same = profile(counts[groups["Natural"]], counts[groups["Natural"]])
assert same["fractional_difference"].abs().max() < 0.2           # same sample as background: differences only from bootstrap noise
print("profile() tests passed")

profile() tests passed


## Run for all groups and both backgrounds

In [5]:
res = []
for name, mask in groups.items():
    fg, rest = counts[mask], counts[~mask]
    a = profile(fg, bg_counts=rest);        a.insert(0, "background", "other peptides")
    b = profile(fg, bg_freq=swissprot);     b.insert(0, "background", "Swiss-Prot 51")
    for r in (a, b):
        r.insert(0, "group", name)
        r.insert(2, "n_peptides", int(mask.sum()))
        res.append(r)
res = pd.concat(res, ignore_index=True)
res.to_csv("../results/enrichment_results.csv", index=False)

vs_rest = res[res.background == "other peptides"]
print("amino acids with a Bonferroni-significant difference from the other peptides (out of 20):")
vs_rest.groupby("group")["significant"].sum().reindex(list(groups))

amino acids with a Bonferroni-significant difference from the other peptides (out of 20):


group
Natural           17
Synthetic         17
Gram+             13
Gram-             12
Fungus            14
Cancer            11
Virus             15
Mammalian cell    19
Name: significant, dtype: object

## Heatmaps: fractional difference of every amino acid in every group
Left: against the **other peptides** (a star marks a Bonferroni-significant difference). Right: against **Swiss-Prot**. Colour = fractional difference *h* (red = more common in the group, blue = less common; colours are capped at ±2).

In [6]:
def matrix(background):
    d = res[res.background == background]
    return (d.pivot(index="group", columns="aa", values="fractional_difference").loc[list(groups), AA_PLOT],
            d.pivot(index="group", columns="aa", values="significant").loc[list(groups), AA_PLOT] if background == "other peptides" else None)

npep = res.groupby("group")["n_peptides"].first().loc[list(groups)]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"wspace": 0.05})
for ax, bgname, title in zip(axes, ["other peptides", "Swiss-Prot 51"],
                             ["vs the other peptides (* = significant, Bonferroni)", "vs Swiss-Prot 51"]):
    mat, sig = matrix(bgname)
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(20)); ax.set_xticklabels(AA_PLOT)
    ax.set_yticks(range(len(groups)))
    ax.set_yticklabels([f"{g} (n={npep[g]})" for g in groups] if ax is axes[0] else [])
    if sig is not None:
        for i in range(mat.shape[0]):
            for j in range(20):
                if sig.values[i, j]:
                    ax.text(j, i, "*", ha="center", va="center", fontsize=12)
    for x in [2.5, 4.5, 8.5, 11.5, 16.5]:
        ax.axvline(x, color="white", lw=2)
    ax.set_title(title, fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.85, label="fractional difference (p - q) / q")
plt.savefig("../figures/enrichment_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Natural vs Synthetic against Swiss-Prot
Bar height = fractional difference from Swiss-Prot 51, error bars as defined by Vacic et al. (2007). DBAASP v2 shows a similar comparison of its ribosomal and synthetic peptides with UniProt (Pirtskhalava et al. 2016, Fig. 2), which is a useful sanity check of our numbers.

In [7]:
COLORS = {"Natural": "#2a9d8f", "Synthetic": "#e76f51"}
fig, ax = plt.subplots(figsize=(11, 3.8))
x = np.arange(20); w = 0.4
for k, g in enumerate(["Natural", "Synthetic"]):
    r = res[(res.group == g) & (res.background == "Swiss-Prot 51")].set_index("aa").loc[AA_PLOT]
    ax.bar(x + (k - 0.5) * w, r["fractional_difference"], w, yerr=r["error_bar"], color=COLORS[g], label=g, capsize=2)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(AA_PLOT)
ax.set_ylabel("fractional difference from Swiss-Prot")
ax.set_title("Natural and Synthetic peptides vs Swiss-Prot 51 (Composition Profiler)", fontsize=10)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("../figures/enrichment_natural_synthetic_swissprot.png", dpi=150)
plt.show()

## Largest differences per group
The three amino acids with the largest positive and the largest negative fractional difference from the other peptides (only Bonferroni-significant ones are listed).

In [8]:
def fmt(d):
    return ", ".join(f"{a} ({v:+.2f})" for a, v in zip(d.aa, d.fractional_difference)) or "-"

rows = []
for g in groups:
    s = vs_rest[(vs_rest.group == g) & vs_rest.significant]
    rows.append({"group": g,
                 "most enriched": fmt(s[s.fractional_difference > 0].nlargest(3, "fractional_difference")),
                 "most depleted": fmt(s[s.fractional_difference < 0].nsmallest(3, "fractional_difference"))})
top = pd.DataFrame(rows).set_index("group")
top.to_csv("../results/enrichment_top.csv")
top

,most enriched,most depleted
group,,
Natural,"C (+2.17), D (+0.95), G (+0.90)","W (-0.72), R (-0.51), K (-0.40)"
Synthetic,"W (+2.59), R (+1.05), K (+0.66)","C (-0.68), D (-0.49), G (-0.47)"
Gram+,"G (+0.25), L (+0.12), W (+0.12)","E (-0.48), Q (-0.35), D (-0.30)"
Gram-,"K (+0.32), P (+0.25), R (+0.18)","E (-0.52), D (-0.35), Q (-0.33)"
Fungus,"C (+0.63), D (+0.28), G (+0.27)","W (-0.42), R (-0.16), K (-0.15)"
Cancer,"W (+0.19), F (+0.17), R (+0.11)","E (-0.25), H (-0.24), D (-0.17)"
Virus,"E (+2.42), Q (+1.29), D (+1.18)","P (-0.41), K (-0.38), R (-0.36)"
Mammalian cell,"K (+0.35), W (+0.34), L (+0.30)","E (-0.45), Y (-0.37), D (-0.36)"


## Natural vs Synthetic: residue table

In [9]:
n = vs_rest[vs_rest.group == "Natural"].set_index("aa").loc[AA_PLOT]
sp_n = res[(res.group == "Natural") & (res.background == "Swiss-Prot 51")].set_index("aa").loc[AA_PLOT]
sp_s = res[(res.group == "Synthetic") & (res.background == "Swiss-Prot 51")].set_index("aa").loc[AA_PLOT]
tab = pd.DataFrame({"pct_Natural": n["pct_group"], "pct_Synthetic": n["pct_background"],
                    "h_Natural_vs_Synthetic": n["fractional_difference"], "p_bonferroni": n["p_bonferroni"],
                    "pct_SwissProt": sp_n["pct_background"],
                    "h_Natural_vs_SwissProt": sp_n["fractional_difference"],
                    "h_Synthetic_vs_SwissProt": sp_s["fractional_difference"]})
tab.round(3)

,pct_Natural,pct_Synthetic,h_Natural_vs_Synthetic,p_bonferroni,pct_SwissProt,h_Natural_vs_SwissProt,h_Synthetic_vs_SwissProt
aa,,,,,,,
K,9.972,16.590,-0.399,0.000,5.925,0.683,1.799
R,5.828,11.979,-0.513,0.000,5.405,0.078,1.217
H,2.101,2.078,0.011,1.000,2.292,-0.083,-0.092
D,2.656,1.364,0.947,0.000,5.355,-0.504,-0.745
E,2.556,1.787,0.431,0.000,6.676,-0.617,-0.733
S,5.636,3.479,0.620,0.000,6.836,-0.176,-0.491
T,3.901,2.359,0.653,0.000,5.415,-0.280,-0.564
N,3.686,2.137,0.725,0.000,4.134,-0.108,-0.483
Q,2.522,2.155,0.170,0.000,3.954,-0.362,-0.455


# Findings and limitations

Fractional difference *h* = (p − q) / q: 0 = same as the background, +1 = twice as common, −0.5 = half as common. Backgrounds: the other non-redundant peptides (t-test with Bonferroni correction) and Swiss-Prot 51 (Vacic et al. 2007, Table 1; no test). With about 14,000 to 155,000 residues per group, 11–19 of the 20 amino acids are "significant" in every group against the other peptides, so the **size of *h*** is what matters.

## Check against the literature
Against Swiss-Prot, the natural peptides (ribosomal + non-ribosomal) have more cysteine (C: h = +3.38, i.e. 4.4 times as common), lysine (+0.68) and glycine (+0.60) and less aspartate (−0.50) and glutamate (−0.62). The synthetic peptides have more tryptophan (W: +4.16), lysine (+1.80) and arginine (+1.22) and less aspartate (−0.75) and glutamate (−0.73). DBAASP v2 reports the same qualitative picture when it compares its own ribosomal and synthetic peptides with UniProt: increased Lys, Cys and Gly and fewer Asp and Glu for ribosomal peptides, increased Lys, Arg and Trp for synthetic ones (Pirtskhalava et al. 2016). The agreement supports the calculation.

## Natural vs Synthetic (2,301 vs 7,650 peptides; each is the other's background)
* **More common in natural peptides than in synthetic ones:** cysteine (6.6% vs 2.1% of residues, h = +2.17), aspartate (+0.95), glycine (11.1% vs 5.9%, +0.90), asparagine (+0.72), threonine (+0.65), serine (+0.62) and proline (+0.55). Cysteine fits natural peptides that are stabilised by disulfide bonds.
* **Less common in natural peptides (= more common in synthetic ones):** tryptophan (1.6% vs 5.8%, h = −0.72; in the other direction tryptophan is 3.6 times as common in synthetic peptides, h = +2.59), arginine (5.8% vs 12.0%, −0.51) and lysine (10.0% vs 16.6%, −0.40). This is the "cationic, tryptophan-rich" pattern of designed peptides, and it agrees with the higher charge per residue in notebook 04.
* Histidine, isoleucine and valine barely differ (|h| ≤ 0.06, not significant). Note that *h* depends on which group is the background, so it is not symmetric between the two groups.

## Target groups (each against the other peptides; "listed with" the group)
* **Virus (n = 682):** glutamate (6.0% of residues, h = +2.42), glutamine (+1.29), aspartate (+1.18), threonine (+0.65) and asparagine (+0.59) are higher; proline (−0.41), lysine (−0.38) and arginine (−0.36) are lower. Peptides listed with viruses are acidic and polar rather than cationic, which matches their median net charge of about 0 in notebook 04. Against Swiss-Prot they are the group closest to an average protein for D and E (D −0.32, E −0.11).
* **Fungus (n = 2,924):** cysteine (+0.63) is higher, tryptophan (−0.42) is lower.
* **Gram+ (n = 7,859) and Gram- (n = 8,028):** differences are modest. Acidic residues are lower (Gram+: E −0.48, Q −0.35, D −0.30; Gram-: E −0.52, D −0.35, Q −0.33), and lysine and arginine are somewhat higher in Gram- (K +0.33, R +0.18; Gram+: K +0.07, R +0.10). These two groups make up about 80% of all peptides, so their "other peptides" background is a small, mixed set, and part of the pattern mirrors the virus result.
* **Cancer (n = 1,819):** small differences only: glutamate (−0.25), histidine (−0.24), tryptophan (+0.19), phenylalanine (+0.17).
* **Mammalian cell (n = 5,469):** lower in glutamate (−0.45), tyrosine (−0.37), cysteine and aspartate (both −0.36); higher in lysine (+0.35), tryptophan (+0.34) and leucine (+0.30). "Listed with mammalian cells" is not the same as toxic.
* **Against Swiss-Prot,** lysine, arginine and tryptophan are more common and aspartate and glutamate less common in essentially every group (right panel of the heatmap), which is the general amino-acid signature of these peptides.

## Limitations
* The t-test treats every residue as independent, as in the original method, but residues of one peptide are not; p-values are therefore optimistic. The bootstrap error bars over whole peptides are the more realistic measure of uncertainty. The plotted error bars are small because each group contains thousands of peptides.
* Composition is pooled over residues, so long peptides contribute more residues than short ones, and no correction is made for length or residue position.
* The result depends on the background. For large or heterogeneous groups (Gram+, Gram-) the other peptides are a small specific set, so results mean "this group vs the rest of DBAASP", and groups overlap so the target results are not independent of each other.
* Swiss-Prot 51 (2006) is an old and rough reference, and DBAASP notes that comparisons with UniProt are most meaningful for ribosomal peptides, which are evolved sequences; synthetic peptides are designed (Pirtskhalava et al. 2021).
* "Listed with" a target group does not mean the peptide is active against it (Pirtskhalava et al. 2021).
* Redundancy removal with CD-HIT at 90% identity (notebook 02) matters for this analysis, but CD-HIT keeps the longest sequence of a cluster and merges shorter peptides into longer ones.

## References
* Vacic V, Uversky VN, Dunker AK, Lonardi S (2007) Composition Profiler: a tool for discovery and visualization of amino acid composition differences. *BMC Bioinformatics* 8:211. https://doi.org/10.1186/1471-2105-8-211 (PMID 17578581). Read in full; source of the fractional difference, the bootstrap over whole proteins, the error bar, the indicator-variable t-test, the Bonferroni option, the choice of background samples, and the Swiss-Prot 51 composition (Table 1).
* Pirtskhalava M, et al. (2016) DBAASP v.2. *Nucleic Acids Res* 44(D1):D1104-D1112. https://doi.org/10.1093/nar/gkv1174 (PMID 26578581). Comparison of DBAASP peptides with the UniProt composition (Fig. 2).
* Pirtskhalava M, et al. (2021) DBAASP v3. *Nucleic Acids Res* 49(D1):D288-D297. https://doi.org/10.1093/nar/gkaa991 (PMID 33151284). "Comparative analysis of AMP composition" tool and the note on ribosomal peptides; tested peptides are included whether active or not.